In [ ]:
# KodeKloud Airlines Text Embedding Lab (STUDENT)
# Instructions: fill the 4 <fill_in the line here> placeholders, then run cells top-to-bottom.


In [ ]:
import re
from pathlib import Path

# <fill_in the line here>
# (Hint: import the vector database client used in this repo)
<fill_in the line here>

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


In [ ]:
BASE_DIR = Path.cwd()

# <fill_in the line here>
# (Hint: set POLICY_PATH to the policy markdown file inside this folder)
POLICY_PATH = <fill_in the line here>

policy_text = POLICY_PATH.read_text(encoding="utf-8")
print(f"Loaded policy: {POLICY_PATH.name} ({len(policy_text):,} chars)")


In [ ]:
def chunk_by_headings(md: str) -> list[dict]:
    parts = re.split(r"(?m)^((?:##|###)\s+.+)$", md)
    chunks: list[dict] = []
    current_title = "(start)"

    for part in parts:
        part = part.strip()
        if not part:
            continue

        if part.startswith("## ") or part.startswith("### "):
            current_title = part.lstrip("# ").strip()
            continue

        text = part.strip()
        if len(text) < 60:
            continue

        chunks.append({"section": current_title, "text": text})

    return chunks

chunks = chunk_by_headings(policy_text)
print("Chunks:", len(chunks))
print("Example section:", chunks[0]["section"])


In [ ]:
# <fill_in the line here>
# (Hint: use SentenceTransformer with the same model as the instructor notebook)
MODEL_NAME = <fill_in the line here>
model = SentenceTransformer(MODEL_NAME)

v = model.encode("What is the cabin baggage limit?")
print("Model:", MODEL_NAME)
print("Vector shape:", v.shape)
print("Vector preview:", np.array2string(v[:10], precision=3))


In [ ]:
DB_PATH = str(BASE_DIR / "lancedb_kodekloud_airline")
TABLE_NAME = "policy_chunks"

# Recreate DB folder each run for a clean demo (optional).
# If you want persistence between runs, comment out the next 2 lines.
if Path(DB_PATH).exists():
    import shutil
    shutil.rmtree(DB_PATH)

db = lancedb.connect(DB_PATH)

texts = [c["text"] for c in chunks]
sections = [c["section"] for c in chunks]

vectors = model.encode(texts, normalize_embeddings=True)
rows = [
    {"section": sections[i], "text": texts[i], "vector": vectors[i].tolist()}
    for i in range(len(texts))
]

if TABLE_NAME in db.table_names():
    db.drop_table(TABLE_NAME)

tbl = db.create_table(TABLE_NAME, data=rows)
print("Rows in table:", tbl.count_rows())


In [ ]:
def search_policy(question: str, k: int = 3) -> pd.DataFrame:
    # <fill_in the line here>
    # (Hint: embed the question using the same normalization setting as the chunks)
    qvec = <fill_in the line here>

    df = tbl.search(qvec).limit(k).to_pandas()
    cols = [c for c in df.columns if c in {"section", "text", "_distance", "score"}]
    return df[cols]

def pretty_print_results(question: str, k: int = 2, preview_chars: int = 500) -> None:
    print("Question:", question)
    results = search_policy(question, k=k)
    for i, row in results.iterrows():
        section = row.get("section", "")
        text = row.get("text", "")
        dist = row.get("_distance", None)
        print("\n--- Match", i + 1, "---")
        if dist is not None:
            print("Distance:", float(dist))
        print("Section:", section)
        print(text[:preview_chars])


In [ ]:
# Q1 (easy)
pretty_print_results("What is the cabin baggage weight limit?", k=2, preview_chars=600)


In [ ]:
# Q2
pretty_print_results("If I cancel 30 hours before departure, what refund do I get?", k=2, preview_chars=600)


In [ ]:
# Q3
pretty_print_results("How can I contact KodeKloud Airlines support?", k=2, preview_chars=600)


In [ ]:
# Q4
pretty_print_results("What is the unaccompanied minor service fee?", k=2, preview_chars=600)


In [ ]:
# Q5 (tough)
pretty_print_results("Can I take a dog in the cabin if my pet + carrier is 10 kg?", k=2, preview_chars=600)


In [ ]:
# Q6 (no shared keywords)
pretty_print_results(
    "When is your helpdesk staffed so I can reach a human for assistance?",
    k=2,
    preview_chars=600,
)
